# CommGuard detector evaluation

This notebook restores the combined evidence bundle, derives deterministic features from complete runs, and evaluates baselines with grouped splits. It must produce a non-empty evaluation artifact before any bounded training-versus-inference performance statement is considered.


## Install a reviewed source revision and restore the corpus

Upload the combined archive from the benign-corpus notebook as a Kaggle Dataset and set `INPUT_BUNDLE` to its read-only `/kaggle/input/...` path. The notebook does not delete working or input directories.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import importlib
import json
import site
import subprocess
import sys
import tarfile

REPO_URL = "https://github.com/waqasm86/CommGuard.git"
GIT_REF = "main"  # Replace with the same reviewed SHA used for collection.
INPUT_BUNDLE = None  # Example: Path("/kaggle/input/commguard-corpus/bundle.tar.gz")
REPO = Path("/kaggle/working/CommGuard")
ARTIFACTS = Path("/kaggle/working/commguard-artifacts")

if REPO.exists() and not (REPO / ".git").is_dir():
    raise RuntimeError(f"Refusing to replace non-Git directory: {REPO}")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", GIT_REF], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-build-isolation", "--no-deps", "-e", str(REPO)],
    check=True,
)
site.addsitedir(str(REPO / "src"))
importlib.invalidate_caches()

def restore_bundle(bundle: Path, destination: Path) -> None:
    if destination.exists() and any(destination.iterdir()):
        raise RuntimeError(f"Refusing to overwrite non-empty artifact directory: {destination}")
    destination.mkdir(parents=True, exist_ok=True)
    destination_root = destination.resolve()
    with tarfile.open(bundle, "r:gz") as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            if target != destination_root and destination_root not in target.parents:
                raise RuntimeError(f"Unsafe archive member: {member.name}")
            if not (member.isfile() or member.isdir()):
                raise RuntimeError(f"Unsupported archive member: {member.name}")
        archive.extractall(destination)

if INPUT_BUNDLE is not None:
    restore_bundle(Path(INPUT_BUNDLE), ARTIFACTS)
elif not ARTIFACTS.exists():
    raise RuntimeError("Set INPUT_BUNDLE to the combined corpus archive from notebook 2.")

summary_path = ARTIFACTS / "benign-corpus-summary.json"
if not summary_path.exists():
    raise RuntimeError("The restored bundle has no benign corpus summary.")
corpus_summary = json.loads(summary_path.read_text(encoding="utf-8"))
if corpus_summary.get("completed", 0) < 18:
    raise RuntimeError("Fewer than 18 corpus runs completed; grouped evaluation is blocked.")
print("Reviewed source commit:", COMMIT)
print("Completed corpus runs:", corpus_summary["completed"])


## Verify the controlled dual-T4 environment

Evaluation uses restored evidence, but strict preflight records that this staged study is still being handled in the specified two-T4 Kaggle environment.


In [ ]:
from commguard.environment import check_environment

environment = check_environment(strict=True, output=ARTIFACTS)
assert environment["readiness"]["exactly_two_gpus"]
assert environment["readiness"]["both_t4"]
print("Strict dual-T4 readiness:", environment["strict_ready"])


## Extract features and evaluate

This step is disabled by default. Set `RUN_DETECTOR_EVALUATION = True` only after inspecting the restored run manifests and calibration status. CommGuard's evaluator requires a fully `supported` calibration unless explicit negative-result analysis is requested.


In [ ]:
from commguard.evaluation import evaluate_detector
from commguard.features import extract_features

RUN_DETECTOR_EVALUATION = False

evaluation = None
if RUN_DETECTOR_EVALUATION:
    features = extract_features(ARTIFACTS, output=ARTIFACTS)
    if not features:
        raise RuntimeError("No feature rows were created from the complete corpus runs.")
    evaluation = evaluate_detector(ARTIFACTS, output=ARTIFACTS)
    if not evaluation:
        raise RuntimeError("Detector evaluation returned an empty result.")
    print("Feature rows:", len(features))
    print("Evaluation artifact created for grouped analysis.")
else:
    print("Evaluation skipped. Inspect the evidence, then set RUN_DETECTOR_EVALUATION=True.")


## Export final evidence

A non-empty export records the evidence and evaluation outputs; it does not by itself justify a performance claim. Interpret only the grouped metrics and documented limitations.


In [ ]:
from commguard.artifacts import ArtifactStore

if evaluation is not None:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archive_path = Path("/kaggle/working") / f"commguard-evaluation-{stamp}-{COMMIT[:12]}.tar.gz"
    exported = ArtifactStore(ARTIFACTS).export(archive_path)
    print("Final evidence archive:", exported)
else:
    print("Nothing exported because evaluation was not enabled.")
